# 🌱 쑥쑥 키움 — 우리 아이 언어 · 식사 성장 도우미 (Gradio 6 + OpenAI)

**AI 업무자동화 과정 2-1 · 4일차 코랩 실습**

만 1~3세 아이를 키우는 부모를 위한 챗봇입니다. 언어놀이 · 편식 코치에게 묻고, 매일 언어 미션 1개와 식사 미션 1개를 받아 실천을 기록합니다.

### 시작하는 방법 (셀 순서)
| 셀 | 하는 일 |
|---|---|
| ① 설치 | Gradio 6 · OpenAI 라이브러리를 설치합니다 (1분) |
| ② 저장 위치 | 기록을 담을 JSON 파일 위치를 정합니다. 드라이브 연결을 허용하면 내일도 이어서 씁니다 |
| ③ API 키 | 코랩 왼쪽 🔑 보안 비밀 `OPENAI_API_KEY` 에서 키를 읽습니다 (없으면 화면 칸에 붙여 넣기) |
| ④ 실행 | 쑥쑥 키움 화면을 띄웁니다. `https://….gradio.live` 를 누르면 새 창에서 크게 열립니다 |

화면에서는 **아이 정보 저장 → 코치에게 질문 → 오늘의 미션 받기 → 실천 체크 · 기록 저장 → 성장 기록 확인** 순서로 씁니다.

## 요청서를 어떻게 반영했나 · 다르게 한 점

| 요청 | 이 노트북에서 |
|---|---|
| 메인 카드 4개 → 기능 이동 | 맨 위 카드 단추를 누르면 그 탭으로 이동 |
| 언어발달 · 놀이 코치 / 편식 개선 · 식사 코치 | 요청서의 답변 순서(행동 이해 → 참고 사항 → 놀이 3가지 → 대화 예시 → 주의점 / 원인 → 접근 → 조리법 → 대화 → 실천 → 주의점)대로 답하게 함 |
| JSON(answer · category · followups) · 꼬리 질문 단추 1~3개 · 대화 기록 유지 · 초기화 | 그대로 구현(대화는 `role` · `content` 형식) |
| 아이 정보 · 알레르기를 모든 답과 미션에 반영 | 모든 요청에 아이 정보를 붙여 보냄. **미션에 주의 음식이 들어 있으면 파이썬이 한 번 더 찾아 ⚠️ 표시** |
| 오늘의 미션(언어 1 · 식사 1) · 최근 미션과 겹치지 않게 · 후기 반영 | 최근 미션 10개와 후기 7일치를 함께 보내 추천 |
| 실천 체크 · 후기 · 새로 접한 식재료 저장 | 「기록 저장」(같은 날 다시 저장하면 덮어씀) |
| 성장 기록(최근 7일 · 횟수 · 막대그래프 · 날짜별 기록 · 최근 식재료 · 후기) | 카드 + 막대그래프 + 표. **점수 · 발달 평가로 바꾸지 않고, 기록이 없으면 숫자를 만들지 않음** |
| Gradio State + JSON 파일 · 백업/복원 · DB 로 옮기기 쉽게 | 저장은 「1. 데이터」 함수(`load_data` · `save_data` · `add_mission` · `upsert_record`)에만 모음 |

**다르게 한 점**
- 「CDC · WHO · 소아청소년과 자료를 우선 참고」 — 이 챗봇은 그 문서를 직접 찾아 읽지는 않습니다. AI 에게 그 기관들의 일반 지침에 맞게 답하고 확실한 내용만 기관 이름을 들라고 지시했습니다. **발달 · 건강이 걱정되면 소아청소년과 상담이 먼저**라고 화면 답에서도 안내합니다.
- 「지원하지 않는 입력 파일」은 백업 복원에서만 생깁니다(쑥쑥 키움 백업 .json 이 아니면 안내 후 멈춤).

## ① 설치 (1분쯤)

In [ ]:
!pip install -q -U gradio openai pandas
import gradio, openai
print("Gradio", gradio.__version__, "· openai", openai.__version__)

## ② 기록 저장 위치 — 구글 드라이브 (권장)
허용 창이 뜨면 허용하세요. `내 드라이브/ssukssuk/data.json` 에 저장합니다. 드라이브 없이 쓰려면 이 셀을 건너뛰세요(런타임이 끝나면 기록이 지워지니 「데이터 백업」으로 받아 두세요).

In [ ]:
import os
from google.colab import drive
drive.mount("/content/drive")
os.environ["SSUK_DATA"] = "/content/drive/MyDrive/ssukssuk/data.json"
print("기록 파일:", os.environ["SSUK_DATA"])

## ③ API 키 불러오기

In [ ]:
import os
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
    print("✅ 보안 비밀에서 API 키를 읽었습니다.")
except Exception:
    print("보안 비밀에 키가 없습니다. 화면 위쪽 「API 키」 칸에 붙여 넣으세요.")

## ④ 쑥쑥 키움 실행
멈추려면 ■(중지), 코드를 고쳤으면 이 셀만 다시 실행하세요.

In [ ]:
# ============================================================
# 쑥쑥 키움 — 우리 아이 언어 · 식사 성장 도우미 (Gradio 6 + OpenAI)
#  - 데이터 처리(불러오기 · 저장 · 기록 추가)는 아래 「1. 데이터」 함수에만 모았습니다.
#    나중에 Supabase 같은 DB 로 옮길 때 이 함수들만 바꾸면 됩니다.
#  - 실천 기록은 「참고용 관찰 기록」입니다. 점수 · 발달 평가로 바꾸지 않습니다.
# ============================================================
import os, re, json, inspect, datetime as dt
import pandas as pd
import gradio as gr
from openai import OpenAI

DATA_PATH = os.environ.get("SSUK_DATA", "/content/ssukssuk/data.json")
MODEL_DEFAULT = "gpt-4.1-mini"
AGES = ["12~17개월", "18~23개월", "24~29개월", "30~35개월", "직접 입력"]
LANG_WORRIES = ["말이 늦음", "단어 위주 표현", "문장 표현", "발음", "대화 및 상호작용", "특별한 고민 없음"]
FOOD_WORRIES = ["채소 편식", "고기 편식", "생선 편식", "과일 편식", "음식 거부", "음식 뱉기", "식사 중 자리 이탈", "특별한 고민 없음"]


# ---------- 1. 데이터 (JSON 파일 · 나중에 DB 로 바꿀 자리) ----------
def empty_data():
    return {"profile": {}, "missions": [], "records": []}

def load_data():
    try:
        with open(DATA_PATH, encoding="utf-8") as f:
            d = json.load(f)
        for k, v in empty_data().items():
            d.setdefault(k, v)
        return d
    except FileNotFoundError:
        return empty_data()
    except Exception as e:
        print("데이터 파일을 읽지 못해 새로 시작합니다:", e)
        return empty_data()

def save_data(d):
    try:
        os.makedirs(os.path.dirname(DATA_PATH) or ".", exist_ok=True)
        tmp = DATA_PATH + ".tmp"
        with open(tmp, "w", encoding="utf-8") as f:
            json.dump(d, f, ensure_ascii=False, indent=1)
        os.replace(tmp, DATA_PATH)            # 쓰는 도중 끊겨도 원본이 깨지지 않게
    except Exception as e:
        raise gr.Error(f"저장하지 못했습니다({e}). 잠시 뒤 다시 눌러 주세요. 계속되면 「데이터 백업」으로 파일을 받아 두세요.")

def add_mission(d, mission):
    d["missions"].append(mission)
    d["missions"] = d["missions"][-60:]
    save_data(d)

def upsert_record(d, rec):
    d["records"] = [r for r in d["records"] if r["date"] != rec["date"]] + [rec]
    d["records"].sort(key=lambda r: r["date"])
    save_data(d)


# ---------- 2. 아이 정보 ----------
def profile_text(p):
    if not p:
        return "아이 정보 없음"
    return (f"연령 {p.get('age','?')} · 언어 고민 {', '.join(p.get('lang', [])) or '없음'} · "
            f"식사 고민 {', '.join(p.get('food', [])) or '없음'} · 잘 먹는 음식/좋아하는 놀이 {p.get('likes') or '모름'} · "
            f"주의 음식/알레르기 {p.get('allergy') or '입력 안 함'}")

def save_profile(age, age_custom, lang, food, likes, allergy, d):
    if not age:
        raise gr.Error("아이 연령을 골라 주세요.")
    if age == "직접 입력":
        if not (age_custom or "").strip():
            raise gr.Error("「직접 입력」을 골랐다면 연령을 적어 주세요. (예: 20개월)")
        age = age_custom.strip()
    d["profile"] = {"age": age, "lang": lang or [], "food": food or [], "likes": (likes or "").strip(),
                    "allergy": (allergy or "").strip(), "updated": dt.date.today().isoformat()}
    save_data(d)
    return d, "✅ 아이 정보를 저장했습니다. 이제 코치에게 묻거나 오늘의 미션을 받아 보세요.\n\n" + profile_text(d["profile"])

def allergy_words(p):
    return [w for w in re.split(r"[,\s/·]+", p.get("allergy") or "") if len(w) >= 1 and w not in ("없음", "모름")]


# ---------- 3. OpenAI ----------
SAFETY = """[안전 · 신뢰 원칙]
- CDC · WHO · 대한소아청소년과학회 · 질병관리청 같은 믿을 만한 기관의 일반 지침에 맞게 설명해. 기관 이름을 들 때는 확실한 내용만 말해.
- 발달 정보는 일반 참고 사항이고 아이마다 속도가 다르다는 점을 밝혀. 특정 행동만으로 발달 지연이나 질환을 진단하지 마.
- 이미 하던 말 · 사회성 · 운동 기술을 잃었거나 걱정이 계속되면 소아청소년과 또는 발달 전문가 상담을 권해.
- 억지로 먹이기 · 혼내기 · 보상으로 협박하기는 권하지 마.
- 알레르기 · 삼킴 곤란 · 반복되는 사레 · 성장 부진 · 탈수가 의심되면 의료진 상담을 안내해.
- 통포도 · 견과류 · 떡 · 방울토마토처럼 질식 위험이 있는 음식은 연령에 맞는 크기 · 형태 주의를 함께 적어.
- 아이의 [주의 음식 · 알레르기]에 적힌 재료는 어떤 추천에도 넣지 마.
- 확인되지 않은 의학 정보를 사실처럼 말하지 마."""

COACH_PROMPT = {
    "언어": """너는 영유아(만 1~3세) 언어발달과 놀이를 돕는 따뜻한 코치야. 부모에게 쉬운 말로 답해.
답(answer)은 마크다운으로 다음 순서로 써:
1) 아이 행동 이해하기 2) 이 연령 언어발달 참고 사항 3) 하루 10~15분 언어놀이 3가지(놀이마다 준비물 · 진행 방법)
4) 부모가 쓸 수 있는 대화 예시(따옴표로 실제 문장) 5) 놀이할 때 주의할 점 6) 아이 반응에 따른 다음 단계""",
    "식사": """너는 영유아(만 1~3세) 편식 · 식습관을 돕는 따뜻한 식사 코치야. 부모에게 쉬운 말로 답해.
답(answer)은 마크다운으로 다음 순서로 써:
1) 편식 행동의 가능한 원인 2) 음식 거부감을 줄이는 접근(단계별) 3) 추천 식재료와 간단한 조리법(연령에 맞는 크기 · 질감)
4) 아이에게 쓸 긍정적인 대화 예시 5) 집에서 실천할 식습관 방법 6) 식사 때 주의할 점""",
}

def get_client(key_box):
    key = (key_box or "").strip() or os.environ.get("OPENAI_API_KEY", "")
    if not key:
        raise gr.Error("OpenAI API 키가 없습니다. 위 「API 키」 칸에 붙여 넣거나, 코랩 왼쪽 🔑 보안 비밀에 OPENAI_API_KEY 로 넣고 노트북 액세스를 켜 주세요.")
    return OpenAI(api_key=key)

def friendly(e):
    s = str(e)
    if "invalid_api_key" in s or "Incorrect API key" in s:
        return "API 키가 맞지 않습니다. 키를 다시 확인해 주세요."
    if "insufficient_quota" in s:
        return "OpenAI 계정 사용 한도(크레딧)가 없습니다. platform.openai.com 의 Billing 을 확인해 주세요."
    if "rate_limit" in s or "429" in s:
        return "요청이 너무 빨라 잠시 막혔습니다. 1분 뒤 다시 눌러 주세요."
    if "Connection" in s or "timed out" in s:
        return "OpenAI 에 연결하지 못했습니다. 인터넷 연결을 확인하고 다시 눌러 주세요."
    return "오류가 났습니다: " + s[:300]

def ask_json(client, model, system, messages):
    try:
        r = client.chat.completions.create(model=model, response_format={"type": "json_object"},
                                           messages=[{"role": "system", "content": system}] + messages)
    except Exception as e:
        raise gr.Error(friendly(e))
    raw = r.choices[0].message.content or ""
    try:
        return json.loads(raw)
    except json.JSONDecodeError:                      # JSON 이 깨져도 멈추지 않고 글자 그대로 보여 줍니다
        m = re.search(r"\{.*\}", raw, re.S)
        try:
            return json.loads(m.group(0)) if m else {"answer": raw}
        except Exception:
            return {"answer": raw}

def to_text(content):
    if isinstance(content, str):
        return content
    if isinstance(content, list):
        return "\n".join(c.get("text", "") if isinstance(c, dict) else str(c) for c in content)
    return str(content or "")


# ---------- 4. 코치 챗봇 (언어 · 식사 공용) ----------
def coach(kind, key_box, model, question, history, d):
    question = (question or "").strip()
    if not question:
        raise gr.Error("질문을 적어 주세요.")
    history = history or []
    if not d.get("profile"):
        gr.Warning("아이 정보를 먼저 저장하면 더 맞춤으로 답합니다(「아이 정보」 탭).")
    system = (COACH_PROMPT[kind] + "\n\n" + SAFETY + f"\n\n[아이 정보] {profile_text(d.get('profile'))}\n\n"
              "반드시 JSON 하나로만 답해: {\"answer\": \"마크다운 답\", \"category\": \"" + kind +
              " 안의 세부 분류(예: 표현 늘리기 · 채소 편식)\", \"followups\": [\"이어서 물어볼 질문 1~3개\"]}")
    msgs = [{"role": m["role"], "content": to_text(m["content"])[:2500]} for m in history[-6:]]
    msgs.append({"role": "user", "content": question})
    data = ask_json(get_client(key_box), model, system, msgs)
    ans = (data.get("answer") or "답을 만들지 못했습니다. 다시 한 번 눌러 주세요.").strip()
    if data.get("category"):
        ans = f"`{data['category']}`\n\n" + ans
    fups = [f for f in (data.get("followups") or []) if isinstance(f, str) and f.strip()][:3]
    history = history + [{"role": "user", "content": question}, {"role": "assistant", "content": ans}]
    btns = [gr.update(value=fups[i], visible=True) if i < len(fups) else gr.update(visible=False) for i in range(3)]
    return ("", history, *btns)


# ---------- 5. 오늘의 육아 미션 ----------
MISSION_PROMPT = """너는 영유아 육아 미션을 짜 주는 코치야. 오늘 집에서 바로 할 수 있는 미션 2개(언어 1 · 식사 1)를 만들어.
- 언어 미션: 5~15분, 아이 연령과 언어 고민에 맞게. 구체적인 순서와 부모가 할 말(실제 문장)을 넣어.
- 식사 미션: 5~10분, 새 식재료를 보고 · 만지고 · 냄새 맡거나 간단히 함께 준비하는 활동. 억지로 먹이지 않는 방법으로.
- [최근 미션]과 겹치지 않게 하고, [최근 후기]에서 아이가 좋아한 것은 살리고 싫어한 것은 피해.
- 미션을 끝냈다고 아이가 꼭 말하거나 먹어야 하는 것은 아니야. 참여 자체를 칭찬하도록 써.
반드시 JSON 하나로만 답해:
{"language": {"title": "", "materials": "", "minutes": 10, "steps": ["", ""], "parent_lines": ["", ""], "expect": "이 활동으로 해 볼 수 있는 경험"},
 "meal": {"title": "", "ingredient": "오늘 만나 볼 식재료", "materials": "", "minutes": 10, "steps": ["", ""], "tips": ["긍정적인 접근 방법"], "caution": "질식 · 알레르기 주의(없으면 빈 글자)"}}"""

def mission_md(m, allergy_hit):
    L, M = m.get("language", {}), m.get("meal", {})
    def lst(xs): return "\n".join(f"{i}. {x}" for i, x in enumerate(xs or [], 1))
    warn = (f"\n\n> ⚠️ 이 미션에 「{', '.join(allergy_hit)}」이(가) 들어 있을 수 있습니다. 아이의 주의 음식이니 「미션 다시 받기」를 눌러 주세요."
            if allergy_hit else "")
    lang = f"""### 🗣️ 오늘의 언어 미션 — {L.get('title','')}
**준비물** {L.get('materials','-')} · **시간** {L.get('minutes','?')}분

{lst(L.get('steps'))}

**이렇게 말해 보세요**
""" + "\n".join(f"- 「{x}」" for x in L.get("parent_lines") or []) + f"\n\n**기대할 수 있는 경험** {L.get('expect','')}"
    meal = f"""### 🥦 오늘의 식사 미션 — {M.get('title','')}
**만나 볼 식재료** {M.get('ingredient','-')} · **준비물** {M.get('materials','-')} · **시간** {M.get('minutes','?')}분

{lst(M.get('steps'))}

**긍정적으로 다가가기**
""" + "\n".join(f"- {x}" for x in M.get("tips") or []) + (f"\n\n> 주의: {M['caution']}" if M.get("caution") else "") + warn + \
        "\n\n_먹지 않아도 괜찮아요. 보고 · 만지고 · 냄새 맡은 것도 좋은 경험입니다._"
    return lang, meal

def new_mission(key_box, model, d):
    p = d.get("profile")
    if not p:
        raise gr.Error("먼저 「아이 정보」 탭에서 아이 정보를 저장해 주세요.")
    recent = [f"{m['date']} 언어:{m['language'].get('title','')} / 식사:{m['meal'].get('title','')}" for m in d["missions"][-10:]]
    notes = [f"{r['date']} 언어 반응:{r.get('lang_note','')} / 식사 반응:{r.get('meal_note','')}" for r in d["records"][-7:]]
    user = (f"[아이 정보] {profile_text(p)}\n[최근 미션]\n" + ("\n".join(recent) or "없음") +
            "\n[최근 후기]\n" + ("\n".join(notes) or "없음") + f"\n오늘 날짜: {dt.date.today()}")
    m = ask_json(get_client(key_box), model, MISSION_PROMPT + "\n\n" + SAFETY, [{"role": "user", "content": user}])
    if not isinstance(m.get("language"), dict) or not isinstance(m.get("meal"), dict):
        raise gr.Error("미션 모양이 맞지 않게 왔습니다. 「미션 다시 받기」를 한 번 더 눌러 주세요.")
    m["date"] = dt.date.today().isoformat()
    text = json.dumps(m["meal"], ensure_ascii=False)
    hit = [w for w in allergy_words(p) if w in text]          # 파이썬이 한 번 더 확인
    add_mission(d, m)
    lang, meal = mission_md(m, hit)
    return d, lang, meal, False, False, "", "", ""

def today_mission(d):
    t = dt.date.today().isoformat()
    ms = [m for m in d["missions"] if m.get("date") == t]
    return ms[-1] if ms else None

def save_today(lang_done, meal_done, lang_note, meal_note, new_foods, d):
    m = today_mission(d)
    if not m:
        raise gr.Error("오늘 받은 미션이 없습니다. 먼저 「오늘의 미션 받기」를 눌러 주세요.")
    rec = {"date": m["date"], "lang_title": m["language"].get("title", ""), "lang_done": bool(lang_done),
           "lang_note": (lang_note or "").strip(), "meal_title": m["meal"].get("title", ""), "meal_done": bool(meal_done),
           "new_foods": [f for f in re.split(r"[,\s/·]+", new_foods or "") if f], "meal_note": (meal_note or "").strip()}
    upsert_record(d, rec)
    return d, f"✅ {rec['date']} 기록을 저장했습니다. 「성장 기록」 탭에서 볼 수 있어요."


# ---------- 6. 성장 기록 (점수 없이 · 있는 기록만) ----------
def growth(d):
    recs = d.get("records", [])
    if not recs:
        return ("### 아직 기록이 없어요\n「오늘의 육아 미션」에서 미션을 받고 실천 여부와 후기를 저장하면 여기에 쌓입니다.",
                pd.DataFrame({"날짜": [], "미션": [], "실천": []}), pd.DataFrame())
    today = dt.date.today()
    days = [(today - dt.timedelta(days=i)).isoformat() for i in range(6, -1, -1)]
    byd = {r["date"]: r for r in recs}
    week = [byd[x] for x in days if x in byd]
    lang_n = sum(r["lang_done"] for r in week)
    meal_n = sum(r["meal_done"] for r in week)
    foods = []
    for r in reversed(recs):
        for f in r.get("new_foods", []):
            if f not in foods:
                foods.append(f)
    notes = [f"- {r['date']} · {r.get('lang_note') or ''} {('/ ' + r['meal_note']) if r.get('meal_note') else ''}"
             for r in reversed(recs) if r.get("lang_note") or r.get("meal_note")][:3]
    md = f"""### 최근 7일 실천 현황 <small>({days[0]} ~ {days[-1]})</small>
<div class="cards">
<div class="card"><b>🗣️ 언어놀이 실천</b><span>{lang_n}번</span><small>기록한 날 {len(week)}일 중</small></div>
<div class="card"><b>🥦 식사 미션 실천</b><span>{meal_n}번</span><small>기록한 날 {len(week)}일 중</small></div>
<div class="card"><b>🧺 최근 만나 본 식재료</b><span style="font-size:1em">{', '.join(foods[:8]) or '아직 없음'}</span></div>
</div>

**최근 후기**
""" + ("\n".join(notes) or "- 아직 후기가 없어요") + \
        "\n\n_실천 횟수는 기록일 뿐, 아이의 발달 점수나 평가가 아닙니다._"
    chart = pd.DataFrame([{"날짜": x[5:], "미션": k, "실천": int(bool(byd.get(x, {}).get(f)))}
                          for x in days for k, f in (("언어놀이", "lang_done"), ("식사 미션", "meal_done"))])
    table = pd.DataFrame([{"날짜": r["date"], "언어놀이": r["lang_title"], "언어 실천": "✅" if r["lang_done"] else "—",
                           "언어 반응 · 후기": r["lang_note"], "식사 미션": r["meal_title"], "식사 실천": "✅" if r["meal_done"] else "—",
                           "새로 접한 식재료": ", ".join(r.get("new_foods", [])), "식사 반응 · 후기": r["meal_note"]}
                          for r in reversed(recs)])
    return md, chart, table


# ---------- 7. 백업 · 복원 ----------
def backup(d):
    save_data(d)
    name = f"/content/ssukssuk_backup_{dt.datetime.now():%Y%m%d_%H%M}.json" if os.path.isdir("/content") \
        else os.path.join(os.path.dirname(DATA_PATH) or ".", f"ssukssuk_backup_{dt.datetime.now():%Y%m%d_%H%M}.json")
    with open(name, "w", encoding="utf-8") as f:
        json.dump(d, f, ensure_ascii=False, indent=1)
    return name

def restore(file, d):
    if not file:
        raise gr.Error("백업 파일(.json)을 올려 주세요.")
    path = file if isinstance(file, str) else file.name
    try:
        with open(path, encoding="utf-8") as f:
            nd = json.load(f)
        assert isinstance(nd, dict) and "records" in nd and "profile" in nd
    except Exception:
        raise gr.Error("쑥쑥 키움 백업 파일이 아닙니다. 「데이터 백업」으로 받은 .json 파일을 올려 주세요.")
    for k, v in empty_data().items():
        nd.setdefault(k, v)
    save_data(nd)
    return nd, f"✅ 복원했습니다. 기록 {len(nd['records'])}일 · 미션 {len(nd['missions'])}개"


# ---------- 8. 화면 ----------
CSS = """
.hero{background:linear-gradient(135deg,#fff1f2,#fef9c3 55%,#dcfce7);border-radius:18px;padding:18px 22px;margin-bottom:6px}
.hero h1{margin:0;font-size:1.8em}.hero p{margin:4px 0 0;color:#57534e}
.cards{display:grid;grid-template-columns:repeat(auto-fit,minmax(170px,1fr));gap:12px;margin:8px 0}
.card{border:1px solid #fde68a;border-radius:14px;padding:12px 14px;background:#fffbeb}
.card b{display:block;font-size:.9em;color:#92400e}.card span{display:block;font-size:1.5em;font-weight:700;margin:3px 0}
.card small{color:#78716c;word-break:keep-all}
.menu button{min-height:70px;font-size:1.05em;border-radius:14px}
"""
THEME = gr.themes.Soft(primary_hue="rose", secondary_hue="amber", neutral_hue="stone")
HOWTO = """**시작하는 방법** ① 아이의 기본 정보를 입력해 주세요 → ② 언어발달 또는 식사 고민을 선택해 주세요 →
③ 궁금한 내용을 입력하고 맞춤 답변을 확인해 주세요 → ④ 오늘의 미션을 실천하고 성장 기록을 남겨 보세요."""

chat_kwargs = {"height": 420}
if "type" in inspect.signature(gr.Chatbot.__init__).parameters:      # Gradio 4~5
    chat_kwargs["type"] = "messages"
blocks_kwargs = {"title": "쑥쑥 키움"}
launch_kwargs = {"share": True, "debug": True}
if "theme" in inspect.signature(gr.Blocks.launch).parameters:         # Gradio 6: 테마 · CSS 는 launch 에
    launch_kwargs.update(theme=THEME, css=CSS)
else:
    blocks_kwargs.update(theme=THEME, css=CSS)

def coach_tab(kind, label, examples, key_box, model, data):
    chat = gr.Chatbot(label=label, **chat_kwargs)
    q = gr.Textbox(label="질문", placeholder=examples[0])
    gr.Markdown("**이어서 물어보기** (누르면 바로 답합니다)")
    with gr.Row():
        fb = [gr.Button(visible=False, size="sm") for _ in range(3)]
    gr.Examples(examples, q)
    reset = gr.Button("대화 초기화", size="sm")
    outs = [q, chat, *fb]
    q.submit(lambda *a: coach(kind, *a), [key_box, model, q, chat, data], outs)
    for b in fb:
        b.click(lambda *a: coach(kind, *a), [key_box, model, b, chat, data], outs)
    reset.click(lambda: ([], *[gr.update(visible=False)] * 3), None, [chat, *fb])

with gr.Blocks(**blocks_kwargs) as demo:
    data = gr.State(load_data())
    gr.HTML('<div class="hero"><h1>🌱 쑥쑥 키움</h1><p>매일 조금씩, 우리 아이의 말과 식사 습관을 함께 키워요!</p></div>')
    gr.Markdown(HOWTO)
    with gr.Row():
        key_box = gr.Textbox(label="API 키 (보안 비밀에 넣었다면 비워 두기)", type="password", scale=3)
        model = gr.Dropdown(["gpt-4.1-mini", "gpt-4o-mini", "gpt-4.1"], value=MODEL_DEFAULT, label="모델",
                            allow_custom_value=True, scale=1)
    with gr.Row(elem_classes="menu"):
        m1 = gr.Button("💬 언어발달 · 놀이 코치")
        m2 = gr.Button("🍚 편식 개선 · 식사 코치")
        m3 = gr.Button("🌱 오늘의 육아 미션")
        m4 = gr.Button("📔 우리 아이 성장 기록")

    with gr.Tabs(selected="profile") as tabs:
        with gr.Tab("아이 정보", id="profile"):
            p0 = load_data()["profile"]
            with gr.Row():
                age = gr.Dropdown(AGES, label="아이 연령", value=p0.get("age") if p0.get("age") in AGES else None)
                age_custom = gr.Textbox(label="직접 입력(예: 20개월)", value=p0.get("age") if p0.get("age") and p0.get("age") not in AGES else "")
            lang = gr.CheckboxGroup(LANG_WORRIES, label="언어발달 고민", value=p0.get("lang", []))
            food = gr.CheckboxGroup(FOOD_WORRIES, label="식사 고민", value=p0.get("food", []))
            likes = gr.Textbox(label="잘 먹는 음식 · 좋아하는 놀이", value=p0.get("likes", ""), placeholder="예: 바나나, 계란찜 / 자동차 놀이, 동물 그림책")
            allergy = gr.Textbox(label="주의해야 할 음식 · 알레르기 (모르면 비워 두기)", value=p0.get("allergy", ""), placeholder="예: 땅콩, 새우")
            p_btn = gr.Button("아이 정보 저장", variant="primary")
            p_msg = gr.Markdown(profile_text(p0) if p0 else "")

        with gr.Tab("💬 언어발달 · 놀이 코치", id="lang"):
            coach_tab("언어", "언어발달 · 놀이 코치",
                      ["2살 아이와 하루 10분 동안 할 수 있는 언어놀이 알려줘.",
                       "아이가 원하는 것을 손가락으로만 가리키는데 어떻게 말하도록 유도할까?",
                       "아이가 단어만 이야기하는데 문장 표현을 늘리는 방법은?",
                       "동물 이름을 활용한 재미있는 말놀이 알려줘.",
                       "그림책을 활용해 자연스럽게 대화를 늘리는 방법은?"], key_box, model, data)

        with gr.Tab("🍚 편식 개선 · 식사 코치", id="food"):
            coach_tab("식사", "편식 개선 · 식사 코치",
                      ["채소를 전혀 먹지 않으려고 해.", "고기만 먹고 다른 반찬은 거부해.",
                       "음식을 입에 넣었다가 계속 뱉어.", "새로운 음식은 절대 먹으려고 하지 않아.",
                       "간식은 잘 먹는데 밥은 잘 먹지 않아.", "브로콜리를 활용한 유아식 3가지 추천해 줘."], key_box, model, data)

        with gr.Tab("🌱 오늘의 육아 미션", id="mission"):
            mi_btn = gr.Button("오늘의 미션 받기 (다시 누르면 새 미션)", variant="primary")
            with gr.Row():
                lang_card = gr.Markdown("오늘의 언어 미션이 여기에 나옵니다.")
                meal_card = gr.Markdown("오늘의 식사 미션이 여기에 나옵니다.")
            gr.Markdown("#### 미션 실천 체크")
            with gr.Row():
                lang_done = gr.Checkbox(label="언어 미션 완료")
                meal_done = gr.Checkbox(label="식사 미션 완료")
            with gr.Row():
                lang_note = gr.Textbox(label="아이의 언어 반응 · 부모 후기", placeholder="예: '멍멍'을 따라 하며 웃었어요")
                meal_note = gr.Textbox(label="식사 반응 · 부모 후기", placeholder="예: 만져 보고 냄새는 맡았는데 먹지는 않았어요")
            new_foods = gr.Textbox(label="오늘 새로 접한 식재료(쉼표로)", placeholder="예: 브로콜리, 당근")
            rec_btn = gr.Button("기록 저장", variant="primary")
            rec_msg = gr.Markdown()

        with gr.Tab("📔 우리 아이 성장 기록", id="growth"):
            g_md = gr.Markdown()
            g_chart = gr.BarPlot(x="날짜", y="실천", color="미션", title="최근 7일 미션 실천(1 = 실천한 날)", y_lim=[0, 1])
            g_tbl = gr.Dataframe(label="날짜별 활동 기록", interactive=False, wrap=True)
            g_btn = gr.Button("새로 고침", size="sm")
            gr.Markdown("#### 데이터 백업 · 복원\n코랩 런타임이 끝나면 파일이 지워질 수 있으니 가끔 백업 파일을 받아 두세요.")
            with gr.Row():
                bk_btn = gr.Button("데이터 백업 (JSON 받기)")
                bk_file = gr.File(label="받을 파일", interactive=False)
            with gr.Row():
                rs_file = gr.File(label="복원할 백업 파일(.json)", file_types=[".json"], type="filepath")
                rs_btn = gr.Button("복원")
            rs_msg = gr.Markdown()

    # 메뉴 카드 → 탭 이동
    for b, tid in ((m1, "lang"), (m2, "food"), (m3, "mission"), (m4, "growth")):
        b.click(lambda t=tid: gr.Tabs(selected=t), None, tabs)
    m4.click(growth, data, [g_md, g_chart, g_tbl])

    p_btn.click(save_profile, [age, age_custom, lang, food, likes, allergy, data], [data, p_msg])
    mi_btn.click(new_mission, [key_box, model, data],
                 [data, lang_card, meal_card, lang_done, meal_done, lang_note, meal_note, new_foods])
    rec_btn.click(save_today, [lang_done, meal_done, lang_note, meal_note, new_foods, data], [data, rec_msg]) \
        .then(growth, data, [g_md, g_chart, g_tbl])
    g_btn.click(growth, data, [g_md, g_chart, g_tbl])
    bk_btn.click(backup, data, bk_file)
    rs_btn.click(restore, [rs_file, data], [data, rs_msg]).then(growth, data, [g_md, g_chart, g_tbl])

    def on_load(d):
        m = today_mission(d)
        cards = mission_md(m, []) if m else ("오늘의 언어 미션이 여기에 나옵니다.", "오늘의 식사 미션이 여기에 나옵니다.")
        return (*growth(d), *cards)
    demo.load(on_load, data, [g_md, g_chart, g_tbl, lang_card, meal_card])

if __name__ == "__main__":
    demo.launch(**launch_kwargs)

## 테스트해 보기

**언어발달 코치 질문 5개** (탭 아래 예시를 눌러도 됩니다)
1. 2살 아이와 하루 10분 동안 할 수 있는 언어놀이 알려줘.
2. 아이가 원하는 것을 손가락으로만 가리키는데 어떻게 말하도록 유도할까?
3. 아이가 단어만 이야기하는데 문장 표현을 늘리는 방법은?
4. 동물 이름을 활용한 재미있는 말놀이 알려줘.
5. 그림책을 활용해 자연스럽게 대화를 늘리는 방법은?

**편식 코치 질문 5개**
1. 채소를 전혀 먹지 않으려고 해.  2. 고기만 먹고 다른 반찬은 거부해.  3. 음식을 입에 넣었다가 계속 뱉어.
4. 새로운 음식은 절대 먹으려고 하지 않아.  5. 브로콜리를 활용한 유아식 3가지 추천해 줘.

**오늘의 미션 시나리오 3개**
1. 아이 정보(24~29개월 · 단어 위주 표현 · 채소 편식) 저장 → 「오늘의 미션 받기」 → 언어 · 식사 카드가 나오는지
2. 알레르기 칸에 「땅콩」을 적고 다시 받기 → 식사 미션에 땅콩이 없는지(있으면 ⚠️ 경고가 뜨는지)
3. 후기에 「동물 소리를 좋아함 / 브로콜리는 만지기만 함」 저장 → 다음 날(또는 다시 받기) 미션이 그 후기를 반영하는지

**성장 기록 저장 · 조회**: 체크 · 후기 · 새 식재료 입력 → 「기록 저장」 → 📔 성장 기록에서 카드 · 그래프 · 표 확인 → 「데이터 백업」으로 JSON 받기 → 「복원」으로 다시 올리기

## 오류가 나면
| 증상 | 해결 |
|---|---|
| 「OpenAI API 키가 없습니다」 | 보안 비밀 이름 `OPENAI_API_KEY` · 노트북 액세스 켜기 → ③ · ④ 다시 실행 |
| 「사용 한도(크레딧)가 없습니다」 | platform.openai.com → Billing |
| 다음 날 기록이 없음 | ② 드라이브 연결 후 ④ 다시 실행, 또는 받아 둔 백업 파일을 「복원」 |
| 「저장하지 못했습니다」 | 잠시 뒤 다시 저장, 계속되면 「데이터 백업」으로 받아 두기 |